# Rerank Latency Test

Learn more about Voyage's Rerank API [here](https://docs.voyageai.com/docs/reranker).

This notebook benchmarks reranking latency for Voyage (rerank-2.5) using text transcriptions from the IRPAPERS dataset.

For each trial, we randomly sample k documents and a query from the dataset, then measure wall-clock time for a single rerank call returning the top 10 results. 

We sweep k across [10, 20, 50, 100, 200, 500] with 5 random samples per k value to control for content-dependent variation and potential server-side caching. All measurements use time.perf_counter() for monotonic precision. 

### TLDR: The Results

| k | Mean (s) | Std (s) | Min (s) | Max (s) |
|---:|---:|---:|---:|---:|
| 10 | 0.255 | 0.063 | 0.215 | 0.367 |
| 20 | 0.302 | 0.080 | 0.239 | 0.438 |
| 50 | 0.396 | 0.073 | 0.319 | 0.512 |
| 100 | 0.667 | 0.052 | 0.603 | 0.716 |
| 200 | 1.234 | 0.103 | 1.136 | 1.391 |
| 500 | 2.578 | 0.178 | 2.339 | 2.768 |

In [ ]:
import os
import time
import random
import statistics
 
import voyageai
import numpy as np
from datasets import load_dataset

os.environ["VOYAGE_API_KEY"] = "YOUR-VOYAGE-API-KEY"

In [3]:
MODEL = "rerank-2.5"
TOP_N = 10
DOC_COUNTS = [10, 20, 50, 100, 200, 500]  # ablation over k
SAMPLES_PER_K = 5                          # random draws per k
SEED = 42
 
vo = voyageai.Client(api_key=os.environ["VOYAGE_API_KEY"])

In [22]:
print("Loading IRPAPERS dataset...")
docs_ds = load_dataset("weaviate/IRPAPERS", "docs", split="train")
queries_ds = load_dataset("weaviate/IRPAPERS", "queries", split="train")

all_docs = [doc["transcription"] for doc in docs_ds]
all_docs = [doc for doc in all_docs if doc is not None]
all_queries = [q["question"] for q in queries_ds]
 
print(f"  {len(all_docs)} docs, {len(all_queries)} queries loaded\n")

Loading IRPAPERS dataset...
  3226 docs, 180 queries loaded



In [24]:
def sample_docs(docs: list[str], k: int, rng: random.Random) -> list[str]:
    """Draw k docs without replacement. Caps at len(docs)."""
    k = min(k, len(docs))
    return rng.sample(docs, k)
 
 
def sample_query(queries: list[str], rng: random.Random) -> str:
    return rng.choice(queries)

In [25]:
def bench_rerank(query: str, documents: list[str], max_retries: int = 3) -> float:
    """Time a single rerank call with retries. Returns wall-clock seconds."""
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            _ = vo.rerank(model=MODEL, query=query, documents=documents, top_k=TOP_N)
            return time.perf_counter() - t0
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"    retry {attempt+1}/{max_retries}: {e}")
                time.sleep(1)
            else:
                raise

In [26]:
def run_ablation(
    all_docs: list[str],
    all_queries: list[str],
    doc_counts: list[int],
    samples_per_k: int,
    seed: int,
) -> list[dict]:
    rng = random.Random(seed)
    results = []

    for k in doc_counts:
        times = []
        for trial in range(samples_per_k):
            docs_sample = sample_docs(all_docs, k, rng)
            query = sample_query(all_queries, rng)
            try:
                elapsed = bench_rerank(query, docs_sample)
            except Exception as e:
                print(f"  k={k:>4d}  trial={trial+1}/{samples_per_k}  FAILED: {e}")
                continue
            times.append(elapsed)
            print(f"  k={k:>4d}  trial={trial+1}/{samples_per_k}  {elapsed:.3f}s")

        if times:
            results.append({
                "k": k,
                "n_ok": len(times),
                "n_fail": samples_per_k - len(times),
                "mean_s": statistics.mean(times),
                "std_s": statistics.stdev(times) if len(times) > 1 else 0.0,
                "min_s": min(times),
                "max_s": max(times),
                "times": times,
            })
        else:
            print(f"  k={k:>4d}  ALL TRIALS FAILED, skipping")

    return results

In [27]:
print(f"Ablating over doc counts: {DOC_COUNTS}")
print(f"  {SAMPLES_PER_K} random samples per k, seed={SEED}\n")
 
results = run_ablation(all_docs, all_queries, DOC_COUNTS, SAMPLES_PER_K, SEED)

Ablating over doc counts: [10, 20, 50, 100, 200, 500]
  5 random samples per k, seed=42

  k=  10  trial=1/5  0.238s
  k=  10  trial=2/5  0.215s
  k=  10  trial=3/5  0.367s
  k=  10  trial=4/5  0.239s
  k=  10  trial=5/5  0.217s
  k=  20  trial=1/5  0.286s
  k=  20  trial=2/5  0.299s
  k=  20  trial=3/5  0.239s
  k=  20  trial=4/5  0.246s
  k=  20  trial=5/5  0.438s
  k=  50  trial=1/5  0.412s
  k=  50  trial=2/5  0.512s
  k=  50  trial=3/5  0.319s
  k=  50  trial=4/5  0.374s
  k=  50  trial=5/5  0.362s
  k= 100  trial=1/5  0.619s
  k= 100  trial=2/5  0.695s
  k= 100  trial=3/5  0.700s
  k= 100  trial=4/5  0.716s
  k= 100  trial=5/5  0.603s
  k= 200  trial=1/5  1.136s
  k= 200  trial=2/5  1.277s
  k= 200  trial=3/5  1.201s
  k= 200  trial=4/5  1.391s
  k= 200  trial=5/5  1.163s
  k= 500  trial=1/5  2.339s
  k= 500  trial=2/5  2.466s
  k= 500  trial=3/5  2.597s
  k= 500  trial=4/5  2.722s
  k= 500  trial=5/5  2.768s


In [29]:
print("\n" + "=" * 60)
print(f"{'k':>6s}  {'mean':>8s}  {'std':>8s}  {'min':>8s}  {'max':>8s}")
print("-" * 60)
for r in results:
    print(f"{r['k']:>6d}  {r['mean_s']:>7.3f}s  {r['std_s']:>7.3f}s  {r['min_s']:>7.3f}s  {r['max_s']:>7.3f}s")
print("=" * 60)


     k      mean       std       min       max
------------------------------------------------------------
    10    0.255s    0.063s    0.215s    0.367s
    20    0.302s    0.080s    0.239s    0.438s
    50    0.396s    0.073s    0.319s    0.512s
   100    0.667s    0.052s    0.603s    0.716s
   200    1.234s    0.103s    1.136s    1.391s
   500    2.578s    0.178s    2.339s    2.768s
